In [ ]:
import jax
jax.config.update("jax_enable_x64", True) # Enable 64-bit precision for higher numerical accuracy.
print(jax.__version__, jax.devices())
import jax.numpy as jnp

import matplotlib.pyplot as plt

# Plot style
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve().parent  # notebook lives in <repo>/non-isothermal-vCANN
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from fenicsx.models.plotstyle import C, SERIF, apply_style, style_axes, style_legend
apply_style()

## Kinematics

In [ ]:
def right_cauchy_green(F):
    return F.T @ F

def isochoric_C(C):
    J = jnp.sqrt(jnp.linalg.det(C)) # linalg is short for linear algebra. jnp.linalg is the part of jax.numpy that contains linear algebra functions. jnp.linalg.det computes the determinant of a matrix.
    return J ** (-2.0/3.0) * C, J
    
def invariants(C_bar):
    I1_bar = jnp.trace(C_bar)
    I2_bar = jnp.trace(jnp.linalg.inv(C_bar)) # tr(C_bar^{-1}), not the polynomial invariant I2_bar = 0.5 * (I1_bar**2 - tr(C_bar^2))
    return I1_bar, I2_bar 


## Projection

It is just the chain rule.
- WHAT WE HAVE: the network gives $\bar S = 2 \partial \Psi/\partial \bar C$. That's a fictitious stress.
- WHAT WE NEED: the physical stress is $S = 2 \partial\Psi/\partial C$.
- THE LINK: $\Psi$ depends on $C$ only through $\bar C = J^{-2/3} C$. So,

$$
\mathbf S = 2\frac{\partial\Psi}{\partial\bar{\mathbf C}} : \frac{\partial\bar{\mathbf C}}{\partial\mathbf C} = J^{-2/3}\,\mathbb P : \bar{\mathbf S},
$$

where, remember

$$
\frac{\partial\bar{\mathbf C}}{\partial\mathbf C} = J^{-2/3}\,\mathbb P^{\mathsf T}, \qquad \mathbb P = \mathbb I - \tfrac13\,\mathbf C^{-1}\otimes\mathbf C, \qquad \mathbb P : \mathbf A = \mathbf A - \tfrac13(\mathbf C:\mathbf A)\,\mathbf C^{-1}.
$$

I am going to return then $S$, taking $A = \bar S$.

`axes=2` tells `jnp.tensordot` how many index pairs to sum over. It pairs the last 2 indices of the first array with the first 2 indices of the second, and sums over both. For two 3 x 3 matrices, that's exactly the double contraction:

$$
\mathbf C : \mathbf A = \sum_{i=1}^{3}\sum_{j=1}^{3} C_{ij}\,A_{ij}
$$

In [ ]:
def project(A, C, J):
    C_inv = jnp.linalg.inv(C)
    return J ** (-2.0/3.0) * (A - jnp.tensordot(C, A, axes=2) / 3.0 * C_inv)


## Parameters of the raw network $\Psi^{\mathcal N}_k$

One can notice that I have kept $W_d$ and $w_T$ apart because in $\psi$ only $W_d$ is constrained to be non-negative, while $w_T$ stays free.

### Parameters as a pytree

A **pytree** is JAX's name for any nesting of Python containers (`dict`, `list`, `tuple`) whose innermost elements, the **leaves**, are arrays. The parameters of one network form a pytree:

```
params = {                          # dict
    "W_defo":    (8, 2) array,         # leaf: weights on the deformation inputs d
    "w_T":    (8,)   array,         # leaf: weights on the temperature T_hat
    "b0":     (8,)   array,         # leaf: first-layer bias
    "layers": [                     # list, one entry per extra hidden layer
        {"W": (8, 8), "b": (8,)},   # dict with 2 leaves
    ],
    "w_out":  (8,)   array,         # leaf: output weights (no output bias)
}
```

First layer, with $\mathbf d = (\bar I_1 - 3,\ \bar I_2 - 3)$:

$$
\mathbf z_1 = \operatorname{softplus}\!\big(\mathbf W_{defo}\,\mathbf d + \mathbf w_T\,\hat T + \mathbf b_0\big)
$$

**Why it matters:** JAX transformations walk through the whole tree.

- `jax.grad(loss)(params)` returns a gradient pytree with **the same structure**: `grads["W_defo"]` has shape `(8, 2)`, with entries $\partial\,\mathrm{loss}/\partial (W_{defo})_{ij}$.
- An optimizer step updates every leaf in one line:
  `jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)`.
- `tau_params`, a list of $N_{\mathrm b}$ such dicts, is also a pytree: its leaves are all the arrays of all networks.

**The stored parameters are unconstrained.** Constraints are applied in the forward pass, not in the init:

- $\Psi$: $\ \mathbf W_{\text{used}} = \operatorname{softplus}(\mathbf W_{\text{stored}}) \ge 0$
- $\tau$: $\ \mathbf W_{\text{used}} = \mathbf W_{\text{stored}}$

That is why one `init_tau_params` serves both kinds of network.

One network with `hidden=(8, 8)`: 6 leaves, $16 + 8 + 8 + 64 + 8 + 8 = 112$ parameters.

In [ ]:
def init_psi_params(key, hidden=(8,8)): 
    keys = jax.random.split(key, len(hidden) + 2) # keys is a list of random keys, one for each layer of the neural network. +2 accounts for the input and output layers. jax.random.split splits a random key into multiple keys.
    params = {
        "W_defo": 0.1 * jax.random.normal(keys[0], (hidden[0], 2)), # W_defo is the weight matrix for the first layer of the neural network. It has shape (hidden[0], 2) because the input layer has 2 neurons (I1_bar and I2_bar) and the first hidden layer has hidden[0] neurons. jax.random.normal generates random numbers from a normal distribution.
        "w_T": 0.1 * jax.random.normal(keys[1], (hidden[0],)), # w_T is the weight matrix for the first layer of the neural network. It has shape (hidden[0],) because the input layer has 1 neuron (T) and the first hidden layer has hidden[0] neurons. jax.random.normal generates random numbers from a normal distribution.    
        "b0": jnp.zeros(hidden[0]), 
        "layers": [],
    }
    for i in range(1, len(hidden)):
        params["layers"].append({
            "W": 0.1 * jax.random.normal(keys[i+1], (hidden[i], hidden[i-1])), # W is the weight matrix for the i-th layer of the neural network. It has shape (hidden[i], hidden[i-1]) because the i-th hidden layer has hidden[i] neurons and the (i-1)-th hidden layer has hidden[i-1] neurons. jax.random.normal generates random numbers from a normal distribution.
            "b": jnp.zeros(hidden[i]), # b is the bias vector for the i-th layer of the neural network. It has shape (hidden[i],) because the i-th hidden layer has hidden[i] neurons.
        })
    params["w_out"] = 0.1 * jax.random.normal(keys[-1], (hidden[-1], )) # w_out is the weight matrix for the output layer of the neural network. It has shape (hidden[-1], 1) because the last hidden layer has hidden[-1] neurons and the output layer has 1 neuron (psi). jax.random.normal generates random numbers from a normal distribution.
    return params


## The raw network $\Psi^{\mathcal N}_{k} (\bar {I}_1, \bar {I}_2, \hat T)$

$$
\mathbf z_1 = \underbrace{\operatorname{softplus}}_{\text{activation}}\Bigl(\underbrace{\operatorname{softplus}(\mathbf W_{\mathbf d}^{\mathrm{raw}})}_{\text{constrained weights} \,>\, 0}\,\mathbf d + \mathbf w_T\,\hat T + \mathbf b_0\Bigr)
$$

The inner `pos(...)` is the weight constraint. It's applied to the stored raw weights to make them positive. It has nothing to do with the neuron; it just turns any real number into a positive one. Any positive map would do the job, for example $e^{w}$ or $w^{2}$. Softplus is a common choice because it grows only linearly and does not blow up like $e^{w}$.

Outer `jax.nn.softplus(...)` is the neuron activation function. Without an activation the network would be of course linear. It has to be softplus, or at least have its properties, because polyconvexity argument needs every layer to be: 

- CONVEX, so convexity survives from layer to layer.
- NON-DECREASING, for the composition rule,
- SMOOTH, because we differentiate twice for $\bar {\mathbb C}_k$. ReLU would give a Hessian that's zero almost everywhere.

$$
\operatorname{softplus}(x) = \ln(1 + e^{x}), \qquad \operatorname{softplus}'(x) = \frac{1}{1 + e^{-x}} > 0, \qquad \operatorname{softplus}''(x) > 0
$$

In [ ]:
positive = jax.nn.softplus
act_fn = jax.nn.softplus

def psi_raw(params, I1_bar, I2_bar, T_hat):
    defo = jnp.array([I1_bar-3.0, I2_bar-3.0]) 
    z = act_fn(positive(params["W_defo"]) @ defo + params["w_T"]*T_hat + params["b0"]) 

    for layer in params["layers"]:
        z = act_fn(positive(layer["W"]) @ z + layer["b"])

    return positive(params["w_out"]) @ z

`psi_raw` = $\Psi^{\mathcal N}$ (the network). `psi_corrected` = $\Psi_k$ or $\Psi_\infty^{\mathrm{iso}}$, depending on the parameters passed.

Note that `psi_raw` takes the invariants as separate scalar arguments, not $\bar C$. That lets us differentiate w.r.t. $\bar {I}_1$ and $\bar {I}_2$ individually in the next cell.

$$
\Psi_k^{\mathcal raw}(\bar I_1, \bar I_2, T) = \tilde{\mathbf w}_{\mathrm{out}}^{\mathsf T}\,\sigma\Bigl(\tilde{\mathbf W}_2\,\sigma\bigl(\tilde{\mathbf W}_{\mathbf d}\,\mathbf d + \mathbf w_T\,\hat T + \mathbf b_0\bigr) + \mathbf b_2\Bigr), \qquad \mathbf {defo} = \begin{pmatrix}\bar I_1 - 3\\ \bar I_2 - 3\end{pmatrix}
$$

## The normalized generating potential $\Psi_k$ and $\bar {S}_k$

In [ ]:
EPS = 1e-3 # eps, stress units; smoothing width of α_k = ε softplus(−δ_k/ε) and β_k = ε softplus(δ_k/ε). Smaller eps leads to sharper transitions between the two branches of the energy function, but can also lead to numerical instability. Larger eps leads to smoother transitions, but can also lead to less accurate results. The choice of eps is a trade-off between accuracy and stability.
I_REF = jnp.array(3.0)
print(I_REF)

def psi_corrected(params, C_bar, T_hat, eps=EPS):
    I1_bar, I2_bar = invariants(C_bar)
    jax.debug.print("I1_bar={a}, I2_bar={b}", a=I1_bar, b=I2_bar)
    A_k, B_k = jax.grad(psi_raw, argnums=(1,2))(params, I_REF, I_REF, T_hat)
    delta_k = A_k - B_k
    alpha_k = eps * act_fn(-delta_k / eps)
    beta_k = eps * act_fn( delta_k / eps)
    return (psi_raw(params, I1_bar, I2_bar, T_hat)
            -psi_raw(params, I_REF, I_REF, T_hat)
            + alpha_k * (I1_bar - 3.0) + beta_k * (I2_bar - 3.0))

def S_bar_k(params, C_bar, T_hat):
    return 2.0 * jax.grad(psi_corrected, argnums=1)(params, C_bar, T_hat)

In [ ]:
# Check cell, corrected network

params = init_psi_params(jax.random.PRNGKey(0)) # PRNG stands for Pseudo-Random Number Generator. jax.random.PRNGKey(0) creates a random key with seed 0. This key is used to initialize the parameters of the neural network. The parameters are initialized using a normal distribution with mean 0 and standard deviation 0.1.   
C_bar_ref = jnp.eye(3)                                     # identity matrix 
print("C_bar_ref=", C_bar_ref)
print("------------------------------")

for T_hat in (-1.0, 0.0, 1.0):
    energy = psi_corrected(params, C_bar_ref, T_hat)
    stress = S_bar_k(params, C_bar_ref, T_hat)
    print("-------------------------------")
    print(f"T_hat={T_hat:.1f}, energy={energy:.10f}, jnp.abs(stress).max()={jnp.abs(stress).max():.10f}")
    print("-------------------------------")

In [ ]:
# Check cell, raw network

def S_bar_raw(params, C_bar, T_hat):
    f = lambda Cbar: psi_raw(params, *invariants(Cbar), T_hat)   # Ψ^N as a function of C̄ only
    return 2.0 * jax.grad(f)(C_bar)

params = init_psi_params(jax.random.PRNGKey(0))
C_bar_ref = jnp.eye(3)                                     # identity matrix 
print("C_bar_ref=", C_bar_ref)
print("------------------------------")

for T_hat in (-1.0, 0.0, 1.0):
    energy = psi_raw(params, *invariants(C_bar_ref), T_hat)
    stress = S_bar_raw(params, C_bar_ref, T_hat)
    print("-------------------------------")
    print(f"T_hat={T_hat:.1f}, energy={energy:.10f}, jnp.abs(stress).max()={jnp.abs(stress).max():.10f}")
    print("-------------------------------")
    print("diagonal of S_bar_raw:", jnp.diag(stress))
    print("-------------------------------")

In [ ]:
# Where does δ_k change sign? 

T_grid = jnp.linspace(-1.0, 1.0, 201)
deltas = []
alphas = []
betas = []
for T_hat in T_grid:
    A_k, B_k = jax.grad(psi_raw, argnums=(1,2))(params, I_REF, I_REF, T_hat)
    delta_k = A_k - B_k
    alpha_k = EPS * act_fn(-delta_k / EPS)
    beta_k = EPS * act_fn( delta_k / EPS)
    deltas.append(delta_k)
    alphas.append(alpha_k)
    betas.append(beta_k)

deltas = jnp.array(deltas)
alphas = jnp.array(alphas)
betas = jnp.array(betas)

T_cross = T_grid[jnp.argmin(jnp.abs(deltas))]  # T_hat where δ_k is closest to zero
print(f"T_cross={T_cross:.4f}, δ_k(T_cross)={deltas[jnp.argmin(jnp.abs(deltas))]:.4e}")


fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(T_grid, deltas, lw=2, label=r"$\delta_k(\hat T)$", ls="--")
ax.plot(T_grid, alphas, lw=2, label=r"$\alpha_k(\hat T)$")
ax.plot(T_grid, betas, lw=2, label=r"$\beta_k(\hat T)$")
ax.axhline(0.0, color="gray", lw=0.8, ls="--")       # zero line: where the sign changes
ax.axvline(T_cross, color="gray", lw=0.8, ls="--")  # vertical line at the crossing point
ax.set_xlabel(r"$\hat T$")
ax.set_ylabel(r"Coefficients")
ax.legend(frameon=True)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("delta_k_vs_T.pdf")
fig.savefig("delta_k_vs_T.png", dpi=1000)
plt.show()

print(jnp.allclose(alphas - betas, -deltas, rtol=0.0, atol=1e-14))   # True